# We will now be cleaning only the phenovue experiment for the June experiments

In [1]:
import polars as pl
from pathlib import Path
import unicodedata
import re
import os
import shutil
from time import sleep
from datetime import datetime

def clean_folder(path: str, delete_folder: bool = False) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
        
    if not delete_folder:
        os.makedirs(path, exist_ok=True)

def clean_column(name):
    replacements = {
        'µ': 'u',
        'μ': 'u',
        '²': '2',
        '³': '3',
    }
    
    for char, replacement in replacements.items():
        name = name.replace(char, replacement)
    
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")
    name = name.strip()
    name = name.strip('_')
    
    return name

def parse_harmony_dataset(root: Path) -> Path:
    """
    Parses a harmony file directory and yields the cell analysis file

    Args:
        root (Path): Top level folder where all the harmony datasets are located

    Yields:
        Path, str: yields a file path to the cell analysis AND the harmony dataset name
    
    **NOTE:** The harmony file structure is:
        
        raw name/
        ├─ Evaluation/
        │  ├─ PlateResult.txt
        │  ├─ Objects_Population - NoBorderObj.txt (this is analysis file)
        │  ├─ .ExportDone_CellListsToTxt
        ├─ indexfile.txt

    """
    
    for f in root.glob("*/*"):
        if f.is_file() and not f.name.startswith(".") and f.name != "PlateResults.txt":
            # print(f)
            yield f, f.relative_to(root).parts[0]


## Raw dataset formatting

In [2]:
in_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
               "Study/2026-06-01_20xRep/2026-06-1CP_20xRep_2h_PHENOV"
               "UE__2026-05-26T16_45_20-Measurement-2")

out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live"
                " Study/formatted_raw_datasets/2026-06-01_PHENOVUE_E"
                "vals-Formatted")
clean_folder(out_path, delete_folder=False)

files = sorted(parse_harmony_dataset(in_path))
for f, eval_name in files:
    print(eval_name)
    
    out_name = out_path / f"{eval_name}.tsv"
    
    with (open(f, "r", encoding='utf-8') as in_file,
          open(out_name, "w", encoding='utf-8') as out_file):
        for i, line in enumerate(in_file):
            curr_line = line.strip()
            
            if i < 9:
                print(f"{curr_line} :: {i}")
            elif i == 9:
                cleaned = "\t".join(clean_column(col) for col in line.strip().split("\t"))
                out_file.write(cleaned + "\n")
            else:
                out_file.write(line.rstrip("\t\n") + "\n")
    
    print("\n\n")

Evaluation1
Database Name	CSC :: 0
Database Link	http://192.168.0.1/ODA/OdaService.asmx :: 1
Evaluation Signature	114b3568-8ae9-4482-8d1a-fcc5212e5f22 :: 2
Plate Name	2026-05-26CP_20xRep_2h :: 3
Measurement	Measurement 2 :: 4
Evaluation	Evaluation1 :: 5
Population	Population - NoBorderObj :: 6
 :: 7
[Data] :: 8



Evaluation2
Database Name	CSC :: 0
Database Link	http://192.168.0.1/ODA/OdaService.asmx :: 1
Evaluation Signature	19aef381-76f5-42f4-b23c-982dce36e054 :: 2
Plate Name	2026-05-26CP_20xRep_2h :: 3
Measurement	Measurement 2 :: 4
Evaluation	Evaluation2 :: 5
Population	Population - NoBorderObj :: 6
 :: 7
[Data] :: 8





## Let's format the TSV files

In [4]:
drop_cols = [
    "Row",
    "Column",
    "Field",
    "Object No",
    "X",
    "Y",
    "Bounding Box",
    "Position X [um]",
    "Position Y [um]",
    "Compound",
    "Concentration",
    "Cell Type",
    "Cell Count"
]

sampling_interval = 2 # hours
timepoint_col = 'Timepoint'

input_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                  "Study/formatted_raw_datasets/2026-06-01_PHENOVUE_Eva"
                  "ls-Formatted")

output_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                   "Study/analysis_ready_data/2026-06-01_Evals-AnalysisReady._.PHENOVUE")

clean_folder(output_path)

for f in input_path.iterdir():
    print(f.name)
    
    lf = pl.scan_csv(f, separator="\t", null_values=["", "NA", "NaN", "nan", "NULL"])
    
    # print(*list(lf.schema.to_python().keys()), sep="\n")
    
    lf = (lf
          .with_columns(
              (
                  pl.col("Row").map_elements(lambda x: chr(64 + x), return_dtype=pl.String) +
                  pl.col("Column").cast(pl.String).str.zfill(2)
              ).alias("WellName")
          )
          .select(["WellName", pl.all().exclude("WellName")])
          .drop(drop_cols)
          .with_columns(
              ((pl.col(timepoint_col) - 1) * sampling_interval).cast(pl.Float64)
          )
        )
    
    out_file = output_path / f.name
    lf.sink_csv(out_file, separator="\t")
    

Evaluation1.tsv
Evaluation2.tsv


## lets stitch the datasets together now!

In [ ]:
# lets figure out if there are any other intersecting wells
eval1_wells = set(
    pl.scan_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-06-01_Evals-AnalysisR"
                "eady._.PHENOVUE/Evaluation1.tsv", separator="\t")
    .select("WellName").unique().collect().to_series().to_list()
)
eval2_wells = set(
    pl.scan_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-06-01_Evals-AnalysisR"
                "eady._.PHENOVUE/Evaluation2.tsv", separator="\t")
    .select("WellName").unique().collect().to_series().to_list()
)

intersecting_wells = eval1_wells & eval2_wells

print(intersecting_wells)

{'I12'}


In [5]:
in_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
               "Study/analysis_ready_data/2026-06-01_Evals-AnalysisR"
               "eady._.PHENOVUE")

out_file = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-06-01_20xRep_PHENOVUE.tsv")

lfs = []
for i, f in enumerate(in_path.iterdir(),start=1):
    curr_lf = (
        pl.scan_csv(f, separator="\t")
    )
    
    if i == 1:
        curr_lf = (
            curr_lf
            .filter(pl.col("WellName") != 'I12')
        )
    lfs.append(curr_lf)

pl.concat(lfs).sink_csv(out_file, separator="\t")